# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *xx*  
**Kaggle challenge:** *xx* (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "*xx*"  

**Author 1 (SCIPER):** *Student Name 1 (xxxxx)*  
**Author 2 (SCIPER):** *Student Name 2 (xxxxx)*  
**Author 3 (SCIPER):** *Student Name 3 (xxxxx)*  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

### Imports :

In [1]:
import os
import sys 
assert (sys.version_info.major == 3) and (sys.version_info.minor == 9)

In [2]:
# if not done yet, install the required packages :
#!pip install -r requirements.txt

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

### Paths :

In [4]:
dataset_dir = "dataset_project_iapr2025"
images_dir = os.path.join(dataset_dir, "train")
train_csv_path = os.path.join(dataset_dir, "train.csv")

In [5]:
df = pd.read_csv(train_csv_path)
df["filename"] = df["id"].apply(lambda x: f"L{x:07d}.JPG")
print(df)

         id  Jelly White  Jelly Milk  Jelly Black  Amandina  Crème brulée  \
0   1000756            2           0            0         0             0   
1   1000763            2           3            3         0             0   
2   1000765            0           0            0         0             0   
3   1000768            0           0            0         0             0   
4   1000772            3           2            1         0             0   
..      ...          ...         ...          ...       ...           ...   
85  1010031            0           0            0         0             0   
86  1010032            0           0            0         0             0   
87  1010035            0           0            0         0             0   
88  1010041            0           0            0         2             0   
89  1010045            1           1            1         1             1   

    Triangolo  Tentation noir  Comtesse  Noblesse  Noir authentique  \
0   

In [6]:
def check_image_sizes(image_dir):
    sizes = set()  # set() only keeps unique values
    
    for filename in os.listdir(image_dir):
        if filename.endswith(".JPG"):
            img_path = os.path.join(image_dir, filename)
            image = Image.open(img_path)
            sizes.add(image.size)  # Adds the size (width, height) to the set
    
    if len(sizes) > 1:
        print("Images are of different sizes:")
        print(sizes)
    else:
        print("Every image has the same size :", sizes)

check_image_sizes(images_dir)

Every image has the same size : {(6000, 4000)}


### Dataset :

In [7]:
class ChocolateDataset(Dataset): # inherits from Dataset class in PyTorch
    def __init__(self, dataframe, images_dir, transform=None):
        self.df = dataframe # dataframe = raw data
        self.images_dir = images_dir # directory where images are stored
        self.transform = transform # transformations to be applied to images
        self.labels = dataframe.iloc[:, 1:-1].values # all columns except the first and last one (id and filename)
        self.filenames = dataframe["filename"].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx): # will be called each time we iterate over the dataset in a dataloader
        img_name = os.path.join(self.images_dir, self.filenames[idx])
        image = Image.open(img_name).convert("RGB")
        
        if self.transform:
            image = self.transform(image) # will be applied each time we iterate over the dataset in a dataloader
        
        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # 13 classes
        return image, label


In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Create the dataset and dataloader WITHOUT normalization
dataset = ChocolateDataset(df, images_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [9]:
def calculate_mean_std(dataloader):
    means = []
    stds = []
    for images, _ in dataloader:
        # Calculate mean and std for each batch
        means.append(torch.mean(images, dim=[0, 2, 3])) # size of the image is (batch_size, channels, height, width) => mean for each channel (RGB), output is (3,)
        stds.append(torch.std(images, dim=[0, 2, 3])) # std for each channel (RGB)
    
    # Mean and std are calculated across all batches
    mean = torch.mean(torch.stack(means), dim=0).numpy()
    std = torch.mean(torch.stack(stds), dim=0).numpy()
    
    return mean, std

# Calculate mean and std with the dataloader
mean, std = calculate_mean_std(dataloader)

print(f"Mean: {mean}")
print(f"Std: {std}")


# Recreate the dataset and dataloader WITH normalization
dataset = ChocolateDataset(df, images_dir, transform=transforms.Normalize(mean=mean, std=std))
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

Mean: [0.6848293  0.66036224 0.6537567 ]
Std: [0.16959643 0.16707774 0.19142401]


### CNN :